<a href="https://colab.research.google.com/github/notshruthii/kd-citation-intent/blob/main/kd_citation_intent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Nov 12 13:40:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
PROJECT_DIR="/content/drive/MyDrive/kd-citation-intent"
os.makedirs(PROJECT_DIR,exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints",exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data",exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/result",exist_ok=True)

Mounted at /content/drive


In [ ]:
from datasets import load_dataset


In [ ]:
from datasets import load_dataset

# Load the auto-converted Parquet version of the dataset directly via its- Hugging Face repository ID
dataset = load_dataset("parquet", data_files={
    "train": "hf://datasets/allenai/scicite@~parquet/default/train/0000.parquet",
    "validation": "hf://datasets/allenai/scicite@~parquet/default/validation/0000.parquet",
    "test": "hf://datasets/allenai/scicite@~parquet/default/test/0000.parquet"
})

print(dataset)

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 2.64MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/625k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['string', 'sectionName', 'label', 'citingPaperId', 'citedPaperId', 'excerpt_index', 'isKeyCitation', 'label2', 'citeEnd', 'citeStart', 'source', 'label_confidence', 'label2_confidence', 'id'],
        num_rows: 8194
    })
    validation: Dataset({
        features: ['string', 'sectionName', 'label', 'citingPaperId', 'citedPaperId', 'excerpt_index', 'isKeyCitation', 'label2', 'citeEnd', 'citeStart', 'source', 'label_confidence', 'label2_confidence', 'id'],
        num_rows: 916
    })
    test: Dataset({
        features: ['string', 'sectionName', 'label', 'citingPaperId', 'citedPaperId', 'excerpt_index', 'isKeyCitation', 'label2', 'citeEnd', 'citeStart', 'source', 'label_confidence', 'label2_confidence', 'id'],
        num_rows: 1859
    })
})


In [ ]:
label_names=["method","background","result"]
for i in range(5):
  row=dataset["train"][i]
  print(row["string"])
  print("->",label_names[row["label"]],"\n")

However, how frataxin interacts with the Fe-S cluster biosynthesis components remains unclear as direct one-to-one interactions with each component were reported (IscS [12,22], IscU/Isu1 [6,11,16] or ISD11/Isd11 [14,15]).
-> background 

In the study by Hickey et al. (2012), spikes were sampled from the field at the point of physiological
robinson et al.: genomic regions influencing root traits in barley 11 of 13
maturity, dried, grain threshed by hand, and stored at −20C to preserve grain dormancy before germination testing.
-> background 

The drug also reduces catecholamine secretion, thereby reducing stress and leading to a modest (10-20%) reduction in heart rate and blood pressure, which may be particularly beneficial in patients with cardiovascular disease.(7) Unlike midazolam, dexmedetomidine does not affect the ventilatory response to carbon dioxide.
-> background 

By clustering with lowly aggressive close kin (King 1989a,b; Viblanc et al. 2010; Arnaud, Dobson & Murie 2012), b

In [ ]:
from collections import Counter

for split in ["train","validation","test"]:
  counts=Counter(dataset[split]["label"])
  total=sum(counts.values())
  print(f"--------{split.upper()}-------")
  for label_id,count in sorted(counts.items()):
    name=label_names[label_id]
    percent=(count/total)*100

    print(label_names[label_id],": ",count,f"({percent:.0f}%)")

--------TRAIN-------
method :  2266 (28%)
background :  4840 (59%)
result :  1088 (13%)
--------VALIDATION-------
method :  255 (28%)
background :  538 (59%)
result :  123 (13%)
--------TEST-------
method :  604 (32%)
background :  996 (54%)
result :  259 (14%)


In [ ]:
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score
import os
if os.path.exists(f"{PROJECT_DIR}/checkpoints/teacher"):
    print("Teacher already trained, skipping.")
else:
  MODEL_NAME = "allenai/scibert_scivocab_uncased"
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

  def tokenize(batch):
    return tokenizer(batch["string"], truncation=True,
                     padding="max_length", max_length=128)

  keep = ["label"]
  drop = [c for c in dataset["train"].column_names if c not in keep]
  tokenized = dataset.map(tokenize, batched=True, remove_columns=drop)

  model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label=dict(enumerate(label_names)),
    label2id={n: i for i, n in enumerate(label_names)})

  def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

  args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/checkpoints/teacher_tmp",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    seed=42,
    report_to="none",
  )

  trainer = Trainer(model=model, args=args,
                  train_dataset=tokenized["train"],
                  eval_dataset=tokenized["validation"],
                  compute_metrics=compute_metrics)

  trainer.train()
  print("TEST:", trainer.evaluate(tokenized["test"]))

# Save the teacher to Drive
  trainer.save_model(f"{PROJECT_DIR}/checkpoints/teacher")
  tokenizer.save_pretrained(f"{PROJECT_DIR}/checkpoints/teacher")




Teacher already trained, skipping.


In [ ]:
import torch
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification)

In [ ]:
teacher_path=PROJECT_DIR+"/checkpoints/teacher"
tokenizer=AutoTokenizer.from_pretrained(teacher_path)
teacher=AutoModelForSequenceClassification.from_pretrained(teacher_path)
teacher=teacher.to("cuda")
teacher.eval()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [ ]:
sentences=dataset["train"]["string"]
labels=dataset["train"]["label"]
print(len(sentences))

8194


In [ ]:
batch_sentences=sentences[:4]
inputs=tokenizer(batch_sentences,truncation=True,padding="max_length",max_length=128,return_tensors="pt")
inputs=inputs.to("cuda")
with torch.no_grad():
  outputs=teacher(**inputs)
print(outputs.logits)

tensor([[-0.1514,  4.0491, -2.8007],
        [ 0.1868,  3.9099, -2.7295],
        [ 0.1354,  3.6614, -2.4434],
        [-0.5842,  4.5813, -2.6904]], device='cuda:0')


In [ ]:
all_logits=[]
batch_size=64
for start in range(0,len(sentences),batch_size):
  batch_sentence=sentences[start:start+batch_size]
  inputs=tokenizer(batch_sentence,truncation=True,padding="max_length",max_length=128,return_tensors="pt")
  inputs=inputs.to("cuda")
  with torch.no_grad():
    outputs=teacher(**inputs)
  all_logits.append(outputs.logits.cpu())
logits=torch.cat(all_logits)
print(logits.shape)


torch.Size([8194, 3])


In [ ]:
import pandas as pd
probs=torch.softmax(logits,dim=1)
df=pd.DataFrame()


In [ ]:
df["label"]=labels
df["logit_method"]=logits[:,0].numpy()
df["logit_background"]=logits[:,1].numpy()
df["logit_result"]=logits[:,2].numpy()
df["probs_method"]=probs[:,0].numpy()
df["probs_background"]=probs[:,1].numpy()
df["probs_result"]=probs[:,2].numpy()
df.to_parquet(PROJECT_DIR+"/data/save_labels.parquet")
print(df.head())


   label  logit_method  logit_background  logit_result  probs_method  \
0      1     -0.151437          4.049115     -2.800728      0.014751   
1      1      0.186761          3.909850     -2.729543      0.023559   
2      1      0.135381          3.661355     -2.443436      0.028520   
3      1     -0.584216          4.581338     -2.690430      0.005674   
4      1     -0.657418          4.781676     -2.576257      0.004322   

   probs_background  probs_result  
0          0.984206      0.001043  
1          0.975165      0.001275  
2          0.969316      0.002164  
3          0.993636      0.000690  
4          0.995044      0.000634  


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

class MetricCalculator:
    def __init__(self, y_true, y_pred, class_names):
        self.y_true = y_true
        self.y_pred = y_pred
        self.class_names = class_names

    def accuracy(self):
        return accuracy_score(self.y_true, self.y_pred)

    def macro_f1(self):
        return f1_score(self.y_true, self.y_pred, average="macro")

    def per_class_f1(self):
        scores = f1_score(self.y_true, self.y_pred, average=None, labels=[0, 1, 2])
        result = {}
        for i in range(len(self.class_names)):
            result[self.class_names[i]] = scores[i]
        return result

    def confusion_matrix(self):
        return confusion_matrix(self.y_true, self.y_pred, labels=[0, 1, 2])

In [ ]:
calc = MetricCalculator([0, 1, 2, 1], [0, 1, 1, 1], label_names)
print(calc.accuracy())
print(calc.macro_f1())
print(calc.per_class_f1())

0.75
0.6
{'method': np.float64(1.0), 'background': np.float64(0.8), 'result': np.float64(0.0)}


In [ ]:
test_sentences = dataset["test"]["string"]
test_labels = dataset["test"]["label"]

test_preds = []
batch_size = 64

for start in range(0, len(test_sentences), batch_size):
    batch = test_sentences[start:start + batch_size]
    inputs = tokenizer(batch, truncation=True, padding="max_length",
                       max_length=128, return_tensors="pt")
    inputs = inputs.to("cuda")
    with torch.no_grad():
        output = teacher(**inputs)
    batch_preds = output.logits.argmax(dim=1).cpu().tolist()
    test_preds = test_preds + batch_preds

print(len(test_preds))

1859


In [ ]:
calc = MetricCalculator(test_labels, test_preds, label_names)
print("accuracy:", calc.accuracy())
print("macro F1:", calc.macro_f1())
print("per class F1:", calc.per_class_f1())
print(calc.confusion_matrix())

accuracy: 0.8676707907477138
macro F1: 0.8544302858833911
per class F1: {'method': np.float64(0.8633217993079585), 'background': np.float64(0.8864097363083164), 'result': np.float64(0.8135593220338984)}
[[499  84  21]
 [ 52 874  70]
 [  1  18 240]]


In [ ]:
import time
import numpy as np
import pandas as pd
import torch

class ModelWrapper:
    def __init__(self, model, tokenizer, name):
        self.model = model.to("cuda")
        self.model.eval()
        self.tokenizer = tokenizer
        self.name = name

    def predict(self, texts):
        inputs = self.tokenizer(texts, truncation=True, padding="max_length",
                                max_length=128, return_tensors="pt").to("cuda")
        with torch.no_grad():
            output = self.model(**inputs)
        return output.logits.cpu()

    def predict_labels(self, texts, batch_size=64):
        all_preds = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            logits = self.predict(batch)
            all_preds = all_preds + logits.argmax(dim=1).tolist()
        return all_preds

    def get_size_mb(self):
        total_bytes = 0
        for p in self.model.parameters():
            total_bytes = total_bytes + p.numel() * p.element_size()
        return total_bytes / 1e6

In [ ]:
def measure_latency(wrapper, texts, batch_size=32, n_warmup=10, n_repeats=100):
    batch = wrapper.tokenizer(texts[:batch_size], truncation=True, padding="max_length",
                              max_length=128, return_tensors="pt").to("cuda")
    times = []
    with torch.no_grad():
        for _ in range(n_warmup):
            wrapper.model(**batch)
        torch.cuda.synchronize()

        for _ in range(n_repeats):
            torch.cuda.synchronize()
            start = time.perf_counter()
            wrapper.model(**batch)
            torch.cuda.synchronize()
            end = time.perf_counter()
            times.append((end - start) * 1000)

    return np.median(times), np.std(times)

In [ ]:
def run_harness(wrapper, dataset, label_names):
    test_texts = dataset["test"]["string"]
    test_labels = dataset["test"]["label"]

    preds = wrapper.predict_labels(test_texts)
    calc = MetricCalculator(test_labels, preds, label_names)
    f1_scores = calc.per_class_f1()

    latency_median, latency_std = measure_latency(wrapper, dataset["validation"]["string"])

    row = {
        "model": wrapper.name,
        "accuracy": float(calc.accuracy()),
        "macro_f1": float(calc.macro_f1()),
        "f1_method": float(f1_scores["method"]),
        "f1_background": float(f1_scores["background"]),
        "f1_result": float(f1_scores["result"]),
        "size_mb": wrapper.get_size_mb(),
        "latency_ms": float(latency_median),
        "latency_std": float(latency_std),
    }
    return row

In [ ]:
teacher_wrapper = ModelWrapper(teacher, tokenizer, "teacher_scibert")
row = run_harness(teacher_wrapper, dataset, label_names)

results = pd.DataFrame([row])
results.to_csv(PROJECT_DIR + "/result/results.csv", index=False)
print(results.T)

                             0
model          teacher_scibert
accuracy              0.867671
macro_f1               0.85443
f1_method             0.863322
f1_background          0.88641
f1_result             0.813559
size_mb             439.683084
latency_ms          192.934435
latency_std           3.657317


In [ ]:
texts = dataset["validation"]["string"]

time1, _ = measure_latency(teacher_wrapper, texts)
time2, _ = measure_latency(teacher_wrapper, texts)

print("Run 1:", round(time1, 2), "ms")
print("Run 2:", round(time2, 2), "ms")
print("Difference:", round(abs(time1 - time2) / time1 * 100, 1), "%")

Run 1: 195.26 ms
Run 2: 203.91 ms
Difference: 4.4 %


In [ ]:
soft = pd.read_parquet(PROJECT_DIR + "/data/save_labels.parquet")
train_labels = dataset["train"]["label"]

print("Rows in soft label file:", len(soft))
print("Rows in train set:", len(train_labels))
print("Labels match in same order:", list(soft["label"]) == list(train_labels))

Rows in soft label file: 8194
Rows in train set: 8194
Labels match in same order: True
